In [1]:
#Import libraries 
import numpy as np
import math
import matplotlib.pyplot as plt
from numba import njit
import os 


In [2]:
#Define constants 

pi = np.pi
deg2rad = pi/180
rad2deg = 1/deg2rad


# This is the main script in the steady state version of the North model

# Set physical parameters
S0 = 1354.0 #The solar constant, corresponds to Q_0=338.5 W/m2
L = 1 #the relative strength of the solar constant, 1 = present
A = 203.3 #Infrared flux A-constant. In units W/m2. In North A = 201.4
B = 2.09 #Infrared flux B-constant. In units W/(m2 * K). In North B = 1.45 
Dmag = 0.44 #Infrared flux D-constant. (Absorps B) In units W/(m2 * K). 
Cl = 9.8 #unit W/m2 * yr/K. Heat Capacity of the system
Toffset = 0 
coldstart = 0 #1, then Toffset is set to -40 for a cold initial temperature
hadleyflag = 0 #0: no hadley cell
albedoflag = 1 #1: albedo feedback with T_crit, otherwise a fixed boundary at 72deg

# Define a function that initializes the model and calculates T
# Set model parameters
jmx = 251 #antallet af dataframes i some form
delt = 1/50 
NMAX = 100000
delx = 2/jmx #bredden på x?

# define an array with latitude steps in both x and phi
x = np.linspace(-1, 1, jmx)
phi = np.arcsin(x) * rad2deg




In [3]:
# inputs
N = 1.79 * 10 ** (20) # mol
ppm = 1e-6
M_co2 = 42.44e-3   # kg/mol

Co2_Volcanic = 51.3 * 1e9      # kg / yr
Co2_start    = 427.0       # ppm

Co2_mol = N * Co2_start * ppm # mol
Co2_kg = Co2_mol * M_co2 # kg

Co2_Volcanic_pmmv = Co2_Volcanic / (M_co2 * N) / ppm # ppm / year


time_step = 10000 # years
time_scale = 20_000_000 # years
n_steps = int(time_scale/time_step)
Co2_array = np.zeros(int(time_scale/time_step))
for t in range(0, time_scale, time_step):
    Co2_array[int(t/time_step)] = Co2_kg + Co2_Volcanic * t * 10

In [4]:
CO2_PPM_array = np.arange(0, time_scale, time_step) *  Co2_Volcanic_pmmv + Co2_start

In [5]:
gamma = 20 #dust sensitivty parameter, m^2/kg
c_dust_per_yr = 1.39e7  #1.39e7kg/year
r_earth = 6378 * 10**3
areal_earth = 4 * np.pi * r_earth**2
albedo_ice = (0.6 - 0.3) * np.exp(- gamma * c_dust_per_yr/areal_earth) + 0.3

In [6]:
# Define the latitude dependent insolation function. xs is the latitudal variable
@njit
def Q_lat(xs, L=1.0, S0=1354.0):
    return L * (S0 / 4.0) * (1.0 - 0.241 * (3.0 * np.square(xs) - 1.0))

In [7]:
# Define the albedo function for Model 1
@njit
def albedo_EXP1(T, xs, jmx, albedoflag):
    alb = np.ones(jmx) * 0.3  # baseline Earth's albedo
    if albedoflag == 1:
        # Variable snowline: temperature-dependent
        for j in range(jmx):
            if T[j] <= -10.0:
                alb[j] = 0.6
    else:
        # Fixed snowline at the poles
        for j in range(jmx):
            if abs(xs[j]) >= 0.95:
                alb[j] = 0.6
    return alb

In [8]:
# Define the albedo function for Model 2
@njit
def albedo(T, xs, jmx, albedoflag, time, gamma = 20, c_dust_per_yr = 1.39e7, areal_earth = 511185932522525.5):
    alb = np.ones(jmx) * 0.3  # baseline Earth's albedo
    if albedoflag == 1:
        # Variable snowline: temperature-dependent
        for j in range(jmx):
            if T[j] <= -10.0:
                alb[j] = (0.6 - 0.3) * np.exp(- time * gamma * c_dust_per_yr/areal_earth) + 0.3
    else:
        # Fixed snowline at the poles
        for j in range(jmx):
            if abs(xs[j]) >= 0.95:
                alb[j] = 0.6
                
    return alb

In [9]:
# Define the heat diffusion function
def D_lat(xs, D0=0.44, hadleyflag=False):   
    n = len(xs)
    D = np.empty(n)
    if hadleyflag == 1:  #latitude dependent diffusion with hadley cell
        sin30 = np.sin(30.0 * np.pi / 180.0)
        for i in range(n):
            D[i] = D0 * (1.0 + 9.0 * np.exp(-((xs[i] / sin30) ** 6)))
    else: # constant diffusion without hadley cell
        for i in range(n):
            D[i] = D0
    return D

In [10]:
def A_func(green_has_con):
    """This function takes the greenhouse gas concentration of CO2 and calculates the Infrared flux A-constant. In units W/m2"""
    phi = np.log(green_has_con/300) #unit in parts per million
    return -326.4 + 9.161 * phi - 3.164 * phi**2 + 0.5468 * phi**3

In [11]:
def B_func(green_has_con):
    """This function takes the greenhouse gas concentration of CO2 and calculates the Infrared flux B-constant. In units (m2 * K)"""
    phi = np.log(green_has_con/300) #unit in parts per million
    return 1.953 - 0.04866 * phi + 0.01309 * phi**2 - 0.002577 * phi**3

In [12]:
A_array = A_func(CO2_PPM_array)

B_array = B_func(CO2_PPM_array)

In [13]:

# Define functions used for matrix operations when solving the equations


@njit

def setupfastMh(delx, jmx, D, B, Cl, delt):
    #set up lambda array.
    lam = (1 - np.arange(-1.0, 1.0, delx) * np.arange(-1.0, 1.0 , delx)) / (delx * delx)
    lam = D[0:jmx] * lam
    M = np.zeros((jmx, jmx))
    # do something special at the boundaries
    M[0,0] = -B-lam[1] #first points
    M[0,1] = lam[1]
    M[jmx-1,jmx-2] = lam[jmx-1] # last points
    M[jmx-1,jmx-1]= - B - lam[jmx-1]
    for jj in range(jmx - 2):
        j=jj + 2
        M[j-1, j-2] = lam[j-1]
        M[j-1, j-1]   = -B - (lam[j]+lam[j-1])
        M[j-1, j] = lam[j]
    # add in heat capacities
    M=M/Cl
    # calculate the inverse of M', the matrix operator.
    Mh=M
    return Mh

def setupfastinvM(Mh, jmx, delt):
    M = 0.5 * Mh.copy()
    for j in range(jmx):
        M[j, j] = M[j, j] - 1.0/delt
    invM = np.linalg.inv(M) 
    return invM

In [14]:
# Calculate an array of annual mean insolation for the latitudes steps
S = Q_lat(x, L, S0)  

A = A_array[0] + 273.15 * B_array[0] 
B = B_array[0]
    
# Calculate an array of diffussion constants
xmp = np.arange(-1, 1 + delx, delx)
D = D_lat(xmp, Dmag, hadleyflag)

# define initial T array
T = 0 * (1 - 2 * np.square(x))
Toffset = -40 
T = T + Toffset
Tinit = T

# Calculate global mean temperature
Tglob = np.mean(T) #ok to take the mean when we use x

# Calculate initial albedo
alb = albedo(T, x, jmx, albedoflag, time = 1)

# Initial boundary conditions
src = (1 - alb) * S/Cl - A/Cl
Mh = setupfastMh(delx, jmx, D, B, Cl, delt)
invM = setupfastinvM(Mh, jmx, delt)
h = np.dot(Mh, T) + src

for n in range(NMAX): #Should be NMAX
        Tglob_prev = Tglob
        # calculate src for this loop
        alb = albedo(T, x, jmx, albedoflag, time = 1)
        src = (1 - alb) * S/Cl - A/Cl
        # Calculate new T.
        T = -np.dot(invM, 0.5* (h + src) + T/delt)

        # Calculate h for next loop.
        h = np.dot(Mh, T) + src
        # Check to see if global mean temperature has converged
        Tglob = np.mean(T)
        #print(Tglob)
        Tchange = Tglob - Tglob_prev
        if abs(Tchange)< 1e-12:
            break

# Compute meridional heat flux and its convergence
a = 6.37e+6; # earth radius in meters
Mh = setupfastMh(delx, jmx, D, 0, 1, delt)
invM = setupfastinvM(Mh, jmx, delt)
Dmp = D[0:jmx]
divF = np.dot(Mh, T)
F = -2 * np.pi * a * a * np.sqrt(1 - x*x) * Dmp * np.gradient(T,delx)


In [15]:
#North Model for EXP1
def north_model_EXP1(T_input, A_val, B_val, D_arr, x, Cl, delx, jmx, delt, NMAX, S0,
                L=1.0, hadleyflag=0, albedoflag=1):
    
    A = A_val + B_val * 273.15
    B = B_val
    D = D_arr.copy()
    T = T_input.copy()
    
    # Latitude-dependent insolation
    S = Q_lat(x, L, S0)

    # Initial albedo
    alb = albedo_EXP1(T, x, jmx, albedoflag)

    # Boundary conditions
    src = (1.0 - alb) * S / Cl - A / Cl
    Mh = setupfastMh(delx, jmx, D, B, Cl, delt)
    invM = setupfastinvM(Mh, jmx, delt)
    h = np.dot(Mh, T) + src

    Tglob = np.mean(T)
    
    for n in range(NMAX):
        Tglob_prev = Tglob
        
        # Update albedo & source
        alb = albedo_EXP1(T, x, jmx, albedoflag)
        src = (1.0 - alb) * S / Cl - A / Cl

        # Temperature update
        T = -np.dot(invM, 0.5 * (h + src) + T / delt)

        # Update h
        h = np.dot(Mh, T) + src
        
        # Check convergence
        Tglob = np.mean(T)
        if abs(Tglob - Tglob_prev) < 1e-12:
            break

    # Compute meridional heat flux
    a = 6.37e6
    Mh = setupfastMh(delx, jmx, D, 0.0, 1.0, delt)
    invM = setupfastinvM(Mh, jmx, delt)
    Dmp = D[:jmx]
    divF = np.dot(Mh, T)

    # Finite-difference gradient instead of np.gradient
    gradT = np.empty_like(T)
    gradT[0] = (T[1] - T[0]) / delx
    for i in range(1, jmx - 1):
        gradT[i] = (T[i + 1] - T[i - 1]) / (2.0 * delx)
    gradT[jmx - 1] = (T[jmx - 1] - T[jmx - 2]) / delx

    F = -2.0 * math.pi * a * a * np.sqrt(1.0 - x * x) * Dmp * gradT

    return T, Tglob, F, alb, S

In [16]:
#North Model for EXP2
def north_model(T_input, A_val, B_val, D_arr, x, Cl, delx, jmx, delt, NMAX, S0, time,
                L=1.0, hadleyflag=0, albedoflag=1):
    
    A = A_val + B_val * 273.15
    B = B_val
    D = D_arr.copy()
    T = T_input.copy()
    
    # Latitude-dependent insolation
    S = Q_lat(x, L, S0)

    # Initial albedo
    alb = albedo(T, x, jmx, albedoflag, time)

    # Boundary conditions
    src = (1.0 - alb) * S / Cl - A / Cl
    Mh = setupfastMh(delx, jmx, D, B, Cl, delt)
    invM = setupfastinvM(Mh, jmx, delt)
    h = np.dot(Mh, T) + src

    Tglob = np.mean(T)
    
    for n in range(NMAX):
        Tglob_prev = Tglob
        
        # Update albedo & source
        alb = albedo(T, x, jmx, albedoflag, time)
        src = (1.0 - alb) * S / Cl - A / Cl

        # Temperature update
        T = -np.dot(invM, 0.5 * (h + src) + T / delt)

        # Update h
        h = np.dot(Mh, T) + src
        
        # Check convergence
        Tglob = np.mean(T)
        if abs(Tglob - Tglob_prev) < 1e-12:
            break

    # Compute meridional heat flux
    a = 6.37e6
    Mh = setupfastMh(delx, jmx, D, 0.0, 1.0, delt)
    invM = setupfastinvM(Mh, jmx, delt)
    Dmp = D[:jmx]
    divF = np.dot(Mh, T)

    # Finite-difference gradient instead of np.gradient
    gradT = np.empty_like(T)
    gradT[0] = (T[1] - T[0]) / delx
    for i in range(1, jmx - 1):
        gradT[i] = (T[i + 1] - T[i - 1]) / (2.0 * delx)
    gradT[jmx - 1] = (T[jmx - 1] - T[jmx - 2]) / delx

    F = -2.0 * math.pi * a * a * np.sqrt(1.0 - x * x) * Dmp * gradT

    return T, Tglob, F, alb, S

In [17]:
#Arrays for EXP1
T_array_EXP1 = np.zeros((np.shape(CO2_PPM_array)[0], np.shape(T)[0]))
F_array_EXP1 = np.zeros((np.shape(CO2_PPM_array)[0], np.shape(F)[0]))
Tglob_array_EXP1 = np.zeros(np.shape(CO2_PPM_array))
alb_array_EXP1 = np.zeros((np.shape(CO2_PPM_array)[0], np.shape(alb)[0]))
S_array_EXP1 = np.zeros((np.shape(CO2_PPM_array)[0], np.shape(S)[0]))

In [18]:
#Arrays for EXP2
T_array = np.zeros((np.shape(CO2_PPM_array)[0], np.shape(T)[0]))
F_array = np.zeros((np.shape(CO2_PPM_array)[0], np.shape(F)[0]))
Tglob_array = np.zeros(np.shape(CO2_PPM_array))
alb_array = np.zeros((np.shape(CO2_PPM_array)[0], np.shape(alb)[0]))
S_array = np.zeros((np.shape(CO2_PPM_array)[0], np.shape(S)[0]))

In [20]:
#Experiment 1
# define initial T array
T = 0 * (1 - 2 * np.square(x))
Toffset = -40
T = T + Toffset
Tinit = T
T_up = T

#north_model(T_input = T_up, A_val = A_array[i], B_val = B_array[i], D_arr = D, L=1, hadleyflag=0, albedoflag=1)
for i in range(2000):
    T_0, T_glob_0, F_0, alb_0, S_0 = north_model_EXP1(T_input = T_up, A_val = A_array[i], B_val = B_array[i], D_arr = D, x = x, Cl = Cl, delx = delx, jmx = jmx, delt = delt, NMAX = NMAX, S0 = S0, L=1.0, hadleyflag=0, albedoflag=1)
    T_array_EXP1[i, :] = T_0
    Tglob_array_EXP1[i] = T_glob_0
    F_array_EXP1[i, :] = F_0
    alb_array_EXP1[i, :] = alb_0
    S_array_EXP1[i, :] = S_0
    T_up = T_0

In [21]:
#Experiment 2
time_arr = np.arange(0, time_scale, time_step)
# define initial T array
T = 0 * (1 - 2 * np.square(x))
Toffset = -40
T = T + Toffset
Tinit = T
T_up = T

#north_model(T_input = T_up, A_val = A_array[i], B_val = B_array[i], D_arr = D, L=1, hadleyflag=0, albedoflag=1)
for i in range(2000):
    T_0, T_glob_0, F_0, alb_0, S_0 = north_model(T_input = T_up, A_val = A_array[i], B_val = B_array[i], D_arr = D, x = x, Cl = Cl, delx = delx, jmx = jmx, delt = delt, NMAX = NMAX, S0 = S0, time = time_arr[i], L=1.0, hadleyflag=0, albedoflag=1)
    T_array[i, :] = T_0
    Tglob_array[i] = T_glob_0
    F_array[i, :] = F_0
    alb_array[i, :] = alb_0
    S_array[i, :] = S_0
    T_up = T_0

In [22]:
#Ice Height Model 2
K = 2.1  # W/(m*K)
Q = 50 * 10**(-3) # W/m^2
T_B = 0 # C
T_S_EXP1 = Tglob_array_EXP1 # C

Ice_h_EXP1 = K * (T_B - T_S_EXP1) / Q 
mask = Ice_h_EXP1 < 0
Ice_h_EXP1[mask] = 0



In [23]:
#Ice Height Model 2
T_S = Tglob_array # C

Ice_h = K * (T_B - T_S) / Q 
mask = Ice_h < 0
Ice_h[mask] = 0

In [24]:
print(Ice_h)

[1528.58545057 1508.81267893 1489.89273394 ...    0.            0.
    0.        ]


In [25]:
print(Ice_h_EXP1)

[1528.58545057 1520.76502989 1513.79020132 ...    0.            0.
    0.        ]
